# 05 — Модель ціни: навчання на A + B

Читає `workspace.gold.listing_features` для міст навчання і пише:
- `workspace.gold.price_predictions` — прогноз для **кожного** оголошення (і baseline, і модель);
- `workspace.gold.model_metrics` — MAE на test для кожного міста й разом;
- `workspace.gold.model_feature_importance` — наскільки зростає MAE, якщо «зламати» ознаку;
- файл моделі `price_model_<version>.joblib` у volume `workspace.gold.models`.

**Модель після цього ноутбука заморожена.** Перший прогін на місті C робиться саме цим файлом через `06_score_city`.
Нова версія моделі = новий `model_version` (наприклад `v2`), старий файл не перезаписується.

Спільний код (ознаки, split, baseline, пайплайн) лежить у `model_utils.py` поруч із ноутбуком.

In [ ]:
dbutils.widgets.text("train_cities", "rome,sicily")
dbutils.widgets.text("model_version", "v1")
dbutils.widgets.text("model_dir", "/Volumes/workspace/gold/models")
dbutils.widgets.dropdown("overwrite_model", "no", ["no", "yes"])

TRAIN_CITIES = [c.strip().lower() for c in dbutils.widgets.get("train_cities").split(",") if c.strip()]
MODEL_VERSION = dbutils.widgets.get("model_version").strip()
MODEL_DIR = dbutils.widgets.get("model_dir").rstrip("/")
OVERWRITE = dbutils.widgets.get("overwrite_model") == "yes"
RUN_LABEL = "train_test"

In [ ]:
import os, sys, json, datetime
import numpy as np, pandas as pd, sklearn, joblib
from pyspark.sql import functions as F

sys.path.append(os.getcwd())          # model_utils.py лежить поруч із ноутбуком
import model_utils as mu

CATALOG = "workspace"
FEATURES = f"{CATALOG}.gold.listing_features"
PREDICTIONS = f"{CATALOG}.gold.price_predictions"
METRICS = f"{CATALOG}.gold.model_metrics"
IMPORTANCE = f"{CATALOG}.gold.model_feature_importance"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.gold.models")

model_path = f"{MODEL_DIR}/price_model_{MODEL_VERSION}.joblib"
if os.path.exists(model_path) and not OVERWRITE:
    raise RuntimeError(f"{model_path} вже існує. Модель заморожена: для змін дайте новий model_version "
                       f"або явно поставте overwrite_model = yes")

## 1. Дані й split

У навчання й оцінку йдуть лише рядки `in_model_scope` (є ціна, `outlier_class` ∈ {normal, tail}).
Test = ~20% оголошень **кожного міста**, цілими хостами (`host_id`), seed 42. Split робиться до будь-якого вибору моделі,
і всі числа нижче — тільки з test.

In [ ]:
pdf = spark.table(FEATURES).filter(F.col("city").isin(TRAIN_CITIES)).toPandas()
missing = set(TRAIN_CITIES) - set(pdf["city"].unique())
if missing:
    raise ValueError(f"Немає ознак для {missing} — запустіть 04_gold_features для цих міст")

pdf["in_model_scope"] = mu.in_scope(pdf)
scope = pdf[pdf["in_model_scope"]].copy()
scope["split"] = mu.host_split(scope)
pdf["split"] = scope["split"].reindex(pdf.index).fillna("out_of_scope")

display(pdf.groupby(["city", "split"]).size().unstack(fill_value=0).reset_index())
train, test = scope[scope.split == "train"], scope[scope.split == "test"]

## 2. Baseline і модель

- **Baseline** — медіана ціни за (`room_type`, `accommodates`) на train **того самого міста**; якщо такої комбінації в train немає → медіана за `room_type` → медіана міста.
  Для довідки рахуємо ще `baseline_ab`: те саме правило, але на train обох міст разом (саме воно буде «старим правилом» для міста C).
- **Модель** — gradient boosting (`HistGradientBoostingRegressor`) на `log(price)`, прогноз → `exp()`.
  Ознаки: розмір житла, тип житла, зручності, профіль хоста, long-stay, ліцензія і щільність оголошень навколо (замість координат і назв районів, які не переносяться на нове місто). Міста серед ознак немає.

In [ ]:
city_baselines = {c: mu.fit_baseline(g) for c, g in train.groupby("city")}
baseline_ab = mu.fit_baseline(train)
model = mu.fit_model(train)

pdf["baseline_pred"] = np.nan
for c, b in city_baselines.items():
    m = pdf["city"] == c
    pdf.loc[m, "baseline_pred"] = mu.predict_baseline(b, pdf[m])
pdf["baseline_ab_pred"] = mu.predict_baseline(baseline_ab, pdf)
pdf["model_pred"] = mu.predict_model(model, pdf)

## 3. MAE на test: кожне місто й разом

In [ ]:
t = pdf[pdf["split"] == "test"]
rows = [{"city": c, **mu.metrics_rows(g)} for c, g in t.groupby("city")]
rows.append({"city": "all", **mu.metrics_rows(t)})
metrics = pd.DataFrame(rows)
display(metrics[["city", "n_test", "median_price", "mae_baseline", "mae_model", "improvement_vs_baseline_pct",
                 "median_ae_model", "bias_model", "mape_model_pct"]])

In [ ]:
# Похибка за room_type — щоб бачити, де модель слабша
t2 = t.assign(abs_err_model=(t.model_pred - t.price).abs(), abs_err_baseline=(t.baseline_pred - t.price).abs())
display(t2.groupby(["city", "room_type"]).agg(n=("price", "size"), median_price=("price", "median"),
                                              mae_baseline=("abs_err_baseline", "mean"), mae_model=("abs_err_model", "mean"))
          .round(1).reset_index())

## 4. Які ознаки важливі

Permutation importance на test (5 000 оголошень, 3 повтори): перемішуємо одну ознаку і дивимось, на скільки EUR зростає MAE.
Це пояснення моделі для звіту й презентації.

In [ ]:
rng = np.random.default_rng(mu.SEED)
sample = t.sample(min(5000, len(t)), random_state=mu.SEED)
base_mae = np.mean(np.abs(mu.predict_model(model, sample) - sample.price))
imp = []
for feat in mu.FEATURES:
    deltas = []
    for _ in range(3):
        sh = sample.copy()
        sh[feat] = rng.permutation(sh[feat].values)
        deltas.append(np.mean(np.abs(mu.predict_model(model, sh) - sh.price)) - base_mae)
    imp.append({"feature": feat, "mae_increase_eur": round(float(np.mean(deltas)), 2)})
importance = pd.DataFrame(imp).sort_values("mae_increase_eur", ascending=False)
display(importance)

## 5. Зберегти модель і записати gold-таблиці

In [ ]:
artifact = {
    "model": model, "baseline_ab": baseline_ab, "city_baselines": city_baselines,
    "features": mu.FEATURES, "train_cities": TRAIN_CITIES, "model_version": MODEL_VERSION,
    "sklearn_version": sklearn.__version__, "trained_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "train_rows": int(len(train)), "test_metrics": metrics.to_dict("records"),
}
joblib.dump(artifact, model_path)
print("Збережено:", model_path)

In [ ]:
def save(pdf_out, table, where):
    sdf = spark.createDataFrame(pdf_out)
    w = sdf.write.format("delta").option("mergeSchema", "true")
    if spark.catalog.tableExists(table):
        w.mode("overwrite").option("replaceWhere", where).saveAsTable(table)
    else:
        w.mode("overwrite").partitionBy("city").saveAsTable(table)

def prediction_table(pdf):
    p = pdf[["city", "snapshot_date", "id", "host_id", "name", "listing_url", "neighbourhood_cleansed", "latitude", "longitude",
             "room_type", "property_type", "accommodates", "bedrooms", "in_model_scope", "split", "outlier_class",
             "price", "baseline_pred", "baseline_ab_pred", "model_pred"]].copy()
    p["error"] = p["model_pred"] - p["price"]                  # < 0: модель занижує ціну
    p["abs_error"] = p["error"].abs()
    p["pct_error"] = p["error"] / p["price"]
    p["model_vs_actual"] = np.select(
        [p["price"].isna(), p["pct_error"] < -0.2, p["pct_error"] > 0.2],
        ["no actual price", "under-priced (model < actual by >20%)", "over-priced (model > actual by >20%)"],
        default="within 20%")
    p["model_version"], p["run_label"] = MODEL_VERSION, RUN_LABEL
    p["snapshot_date"] = pd.to_datetime(p["snapshot_date"]).dt.date
    for c in ["price", "baseline_pred", "baseline_ab_pred", "model_pred", "error", "abs_error", "pct_error", "latitude",
              "longitude", "accommodates", "bedrooms"]:
        p[c] = p[c].astype("float64")
    p["in_model_scope"] = p["in_model_scope"].astype(bool)
    for c in ["name", "listing_url", "neighbourhood_cleansed", "room_type", "property_type", "split", "outlier_class"]:
        p[c] = p[c].astype(object).where(p[c].notna(), None)
    return p

cities_sql = ", ".join(f"'{c}'" for c in TRAIN_CITIES + ["all"])
where = f"model_version = '{MODEL_VERSION}' AND run_label = '{RUN_LABEL}' AND city IN ({cities_sql})"

save(prediction_table(pdf), PREDICTIONS, where)
m_out = metrics.assign(model_version=MODEL_VERSION, run_label=RUN_LABEL, eval_set="test",
                       train_cities=",".join(TRAIN_CITIES), created_at=datetime.datetime.now())
save(m_out, METRICS, where)
imp_out = importance.assign(city="all", model_version=MODEL_VERSION, run_label=RUN_LABEL)
save(imp_out, IMPORTANCE, where)

In [ ]:
display(spark.table(PREDICTIONS).filter(F.col("model_version") == MODEL_VERSION)
        .groupBy("run_label", "city", "split").count().orderBy("city", "split"))
display(spark.table(METRICS).filter(F.col("model_version") == MODEL_VERSION))